# UNet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
CLASS_NAME = None          # 'class_01' to train one class; None trains all
IMAGE_SIZE  = 224          # native image resolution — no upscaling overhead
BATCH_SIZE  = 32           # 2× batch; LR scaled by sqrt(2) accordingly
EPOCHS      = 25
LR          = 3e-4         # sqrt(2) × 2e-4 for batch_size=32
VAL_RATIO   = 0.15
P_SYNTH     = 0.4
P_CUTPASTE  = 0.3
SEED        = 42
ENCODER     = 'resnet34'

USE_ATTENTION_GATES = True
USE_TTA             = True
MULTIVIEW_AGGREGATE = True

# ── loss ──────────────────────────────────────────────────────────────────────
USE_FOCAL_LOSS  = True
FOCAL_GAMMA     = 2.0

# ── augmentation ──────────────────────────────────────────────────────────────
P_HFLIP        = 0.5
P_VFLIP        = 0.5
P_COLOR_JITTER = 0.7

# ── LR schedule ───────────────────────────────────────────────────────────────
LR_WARMUP_EPOCHS = 2

# ── pose conditioning ─────────────────────────────────────────────────────────
POSE_ASSIGNMENTS_PATH = f'/content/drive/MyDrive/ADL/challenge/outputs_patchcore_pose/cluster_assignments/pose_assignments.json'
POSE_VIEW_COUNT       = 8

ROOT_DIR             = "/content/drive/MyDrive/ADL/challenge"
DATASET_DIR          = f'{ROOT_DIR}/data/adl-2025-2026-anomaly-detection'
DESC_CSV             = f'{DATASET_DIR}/anomaly_descriptions.csv'
OUTPUT_DIR           = f'{ROOT_DIR}/outputs_v3_fast'
PRED_DIR             = f'{ROOT_DIR}/preds_v3_fast'
SUBMISSION_MODEL_DIR = f'{OUTPUT_DIR}/models_fast'

print('DATASET_DIR:', DATASET_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


DATASET_DIR: /content/drive/MyDrive/ADL/challenge/data/adl-2025-2026-anomaly-detection
OUTPUT_DIR: /content/drive/MyDrive/ADL/challenge/outputs_v3_fast


In [ ]:
import csv, json, math, os, random, re
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import torchvision
from torchvision.models import resnet18, resnet34, resnet50
from torchvision.models.resnet import (
    ResNet18_Weights, ResNet34_Weights, ResNet50_Weights,
)

print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


torch 2.10.0+cu128
torchvision 0.25.0+cu128


In [ ]:
# Load pose cluster assignments (k-means on DINOv2 features per class).
# These give a *meaningful* pose label per image, replacing the random view
# numbers that appear in filenames (view1..5 have no consistent angle mapping).
POSE_DATA: dict = {}
_pa = Path(POSE_ASSIGNMENTS_PATH)
if _pa.exists():
    with open(_pa) as _f:
        POSE_DATA = json.load(_f)
    print(f'Loaded pose assignments for {len(POSE_DATA)} classes: '
          + ', '.join(f'{c}(k={POSE_DATA[c]["k"]})' for c in sorted(POSE_DATA)))
else:
    print(f'[warn] Pose assignments not found at {POSE_ASSIGNMENTS_PATH}')
    print('       FiLM conditioning will use default cluster 0 for all images.')


Loaded pose assignments for 8 classes: class_01(k=4), class_02(k=2), class_03(k=3), class_04(k=8), class_05(k=4), class_06(k=2), class_07(k=2), class_08(k=8)


## Dataset scanning & anomaly description parsing

In [ ]:
_VIEW_RE = re.compile(r'^(?P<stem>.+)_view(?P<view>[1-5])\.png$')

@dataclass(frozen=True)
class Sample:
    image_path: str
    mask_path:  str | None
    class_name: str
    is_anomaly: bool
    view_id:    int
    sample_id:  str

def _parse_view_and_id(filename: str) -> tuple[str, int]:
    m = _VIEW_RE.match(filename)
    if not m:
        raise ValueError(f'Unexpected filename: {filename}')
    return m.group('stem'), int(m.group('view')) - 1

def load_anomaly_descriptions(csv_path: str) -> dict[str, list[dict]]:
    rows: dict[str, list[dict]] = {}
    with open(csv_path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            cls = row['public_class'].strip()
            rows.setdefault(cls, []).append(row)
    return rows

def motifs_from_description(text: str) -> list[str]:
    t = (text or '').lower()
    motifs: list[str] = []
    def add(name, *keys):
        if any(k in t for k in keys):
            motifs.append(name)
    add('scratch',   'scratch', 'groove', 'linear')
    add('stain',     'stain', 'blotchy', 'discolor', 'contamin')
    add('indent',    'depress', 'hollow', 'dimple', 'concave', 'dent')
    add('protrusion','raised', 'bulging', 'protrusion', 'extrusion')
    add('crack',     'crack', 'fissur', 'fract')
    add('fragment',  'fragment', 'broken', 'pieces', 'jagged')
    add('hole',      'hole', 'pitted')
    add('fuzzy',     'fuzzy', 'powder')
    if 'localized visual anomaly' in t or not motifs:
        motifs.append('localized')
    seen: set[str] = set()
    out: list[str] = []
    for m in motifs:
        if m not in seen:
            out.append(m); seen.add(m)
    return out

def build_motifs_for_class(desc_rows: list[dict] | None) -> list[str]:
    if not desc_rows:
        return ['scratch', 'stain', 'localized']
    allowed = {'scratch','stain','indent','protrusion','crack','fragment','hole','fuzzy','localized'}
    out: list[str] = []
    for r in desc_rows:
        for m in motifs_from_description(r.get('description', '')):
            if m in allowed and m not in out:
                out.append(m)
    return out or ['scratch', 'stain', 'localized']

def scan_class_dataset(dataset_dir: str, class_name: str) -> list[Sample]:
    base      = Path(dataset_dir) / class_name
    train_dir = base / 'train'
    gt_base   = base / 'ground_truth_train'
    samples: list[Sample] = []

    good_dir = train_dir / 'good'
    for fn in sorted(os.listdir(good_dir)):
        if not fn.endswith('.png'):
            continue
        stem, view_id = _parse_view_and_id(fn)
        samples.append(Sample(str(good_dir / fn), None, class_name, False, view_id, stem))

    for sub in sorted(os.listdir(train_dir)):
        if not sub.startswith('anomaly_'):
            continue
        img_dir  = train_dir / sub
        mask_dir = gt_base   / sub
        if not img_dir.exists() or not mask_dir.exists():
            continue
        for fn in sorted(os.listdir(img_dir)):
            if not fn.endswith('.png'):
                continue
            stem, view_id = _parse_view_and_id(fn)
            mask_path = mask_dir / fn
            if not mask_path.exists():
                raise FileNotFoundError(f'Missing mask: {mask_path}')
            samples.append(Sample(str(img_dir / fn), str(mask_path), class_name, True, view_id, stem))

    return samples

def split_by_sample_id(samples: list[Sample], val_ratio: float = 0.15,
                        seed: int = 42) -> tuple[list[Sample], list[Sample]]:
    rng = random.Random(seed)
    ids = sorted({s.sample_id for s in samples})
    rng.shuffle(ids)
    val_ids = set(ids[:max(1, int(len(ids) * val_ratio))])
    return [s for s in samples if s.sample_id not in val_ids],            [s for s in samples if s.sample_id in     val_ids]


In [ ]:
# ── helpers ──────────────────────────────────────────────────────────────────
def pil_to_tensor(img: Image.Image) -> torch.Tensor:
    arr = np.asarray(img, dtype=np.float32)
    if arr.ndim == 2:
        arr = arr[:, :, None]
    return torch.from_numpy(arr.transpose(2, 0, 1) / 255.0)

def mask_to_tensor(mask: Image.Image) -> torch.Tensor:
    arr = (np.asarray(mask.convert('L'), dtype=np.float32) / 255.0 > 0.5).astype(np.float32)
    return torch.from_numpy(arr[None, ...])

def foreground_from_image(img: Image.Image, thr: int = 15) -> torch.Tensor:
    g  = np.asarray(img.convert('L'), dtype=np.uint8)
    fg = (g > thr).astype(np.float32)
    return torch.from_numpy(fg[None, ...])


# ── fractal noise ─────────────────────────────────────────────────────────────
def _fractal_noise_2d(h: int, w: int, octaves: int = 4, base_scale: float = 0.15,
                      rng: random.Random | None = None) -> np.ndarray:
    """Pure-numpy fractal (Perlin-like) noise in [0, 1]. No extra dependencies."""
    if rng is None:
        rng = random.Random()
    noise = np.zeros((h, w), dtype=np.float32)
    amp, freq = 1.0, base_scale
    for _ in range(octaves):
        gw = max(2, int(w * freq) + 2)
        gh = max(2, int(h * freq) + 2)
        grid = np.array([rng.gauss(0.0, 1.0) for _ in range(gw * gh)],
                        dtype=np.float32).reshape(gh, gw)
        xs = np.linspace(0, gw - 1, w)
        ys = np.linspace(0, gh - 1, h)
        x0 = np.floor(xs).astype(int).clip(0, gw - 2)
        y0 = np.floor(ys).astype(int).clip(0, gh - 2)
        x1 = (x0 + 1).clip(0, gw - 1)
        y1 = (y0 + 1).clip(0, gh - 1)
        fx = (xs - x0)[None, :]    # (1, w)
        fy = (ys - y0)[:, None]    # (h, 1)
        layer = (  grid[np.ix_(y0, x0)] * (1 - fy) * (1 - fx)
                 + grid[np.ix_(y0, x1)] * (1 - fy) *      fx
                 + grid[np.ix_(y1, x0)] *      fy  * (1 - fx)
                 + grid[np.ix_(y1, x1)] *      fy  *      fx  )
        noise += amp * layer
        amp   *= 0.5
        freq  *= 2.0
    mn, mx = noise.min(), noise.max()
    return (noise - mn) / (mx - mn + 1e-8)


# ── CutPaste bank ─────────────────────────────────────────────────────────────
class AnomalyCropBank:
    """
    Pre-loads (img_np, mask_np) crops from real anomaly training samples.
    Used during training to paste realistic defect patches onto good images.
    """
    MIN_PIXELS = 50

    def __init__(self, samples: list, image_size: int):
        self.crops: list[tuple[np.ndarray, np.ndarray]] = []
        for s in samples:
            if not s.is_anomaly or s.mask_path is None:
                continue
            img  = Image.open(s.image_path).convert('RGB').resize(
                       (image_size, image_size), Image.BILINEAR)
            mask = Image.open(s.mask_path).convert('L').resize(
                       (image_size, image_size), Image.NEAREST)
            img_np  = np.asarray(img,  dtype=np.float32)
            mask_np = (np.asarray(mask, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)
            if mask_np.sum() >= self.MIN_PIXELS:
                self.crops.append((img_np, mask_np))
        print(f'  AnomalyCropBank: {len(self.crops)} crops')

    def __len__(self) -> int:
        return len(self.crops)

    def sample(self, rng: random.Random) -> tuple[np.ndarray, np.ndarray] | None:
        return rng.choice(self.crops) if self.crops else None


def apply_cutpaste(dst_img: np.ndarray, dst_mask: np.ndarray,
                   src_img: np.ndarray, src_mask: np.ndarray,
                   rng: random.Random) -> tuple[np.ndarray, np.ndarray]:
    """Paste a real anomaly crop at a random position onto dst_img."""
    h, w = dst_img.shape[:2]
    scale  = rng.uniform(0.7, 1.3)
    new_h, new_w = max(4, int(h * scale)), max(4, int(w * scale))

    src_pil   = Image.fromarray(np.clip(src_img, 0, 255).astype(np.uint8)).resize(
                    (new_w, new_h), Image.BILINEAR)
    smask_pil = Image.fromarray((src_mask * 255).astype(np.uint8)).resize(
                    (new_w, new_h), Image.NEAREST)
    src_np  = np.asarray(src_pil,   dtype=np.float32)
    smask   = (np.asarray(smask_pil, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)

    dy = rng.randint(-new_h // 4, h - new_h // 4)
    dx = rng.randint(-new_w // 4, w - new_w // 4)

    sy1 = max(0, -dy);  sy2 = min(new_h, h - dy)
    sx1 = max(0, -dx);  sx2 = min(new_w, w - dx)
    dy1 = max(0,  dy);  dy2 = min(h, dy + new_h)
    dx1 = max(0,  dx);  dx2 = min(w, dx + new_w)

    if sy2 <= sy1 or sx2 <= sx1:
        return dst_img, dst_mask

    patch = src_np[sy1:sy2, sx1:sx2]
    pmask = smask[sy1:sy2, sx1:sx2]
    m = pmask[..., None]

    result_img  = dst_img.copy()
    result_mask = dst_mask.copy()
    result_img[dy1:dy2, dx1:dx2]  = result_img[dy1:dy2, dx1:dx2] * (1 - m) + patch * m
    result_mask[dy1:dy2, dx1:dx2] = np.maximum(result_mask[dy1:dy2, dx1:dx2], pmask)
    return result_img, result_mask


# ── improved synthesize_anomaly ───────────────────────────────────────────────
def _draw_random_blob(draw: ImageDraw.ImageDraw, w: int, h: int,
                      rng: random.Random) -> None:
    for _ in range(rng.randint(3, 10)):
        rx = rng.randint(max(2, w // 30), max(4, w // 10))
        ry = rng.randint(max(2, h // 30), max(4, h // 10))
        cx, cy = rng.randint(0, w - 1), rng.randint(0, h - 1)
        draw.ellipse((cx - rx, cy - ry, cx + rx, cy + ry), fill=255)


def synthesize_anomaly(img: Image.Image, motif: str,
                       rng: random.Random) -> tuple[Image.Image, Image.Image]:
    """
    Generate a synthetic anomaly on *img*.
    stain / fuzzy / localized now use fractal-noise masks for more realistic
    irregular shapes; other motifs are unchanged from v1.
    """
    w, h = img.size
    mask_arr = np.zeros((h, w), dtype=np.float32)

    if motif == 'scratch':
        mask_pil = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask_pil)
        for _ in range(rng.randint(1, 4)):
            x0, y0 = rng.randint(0, w-1), rng.randint(0, h-1)
            x1, y1 = rng.randint(0, w-1), rng.randint(0, h-1)
            draw.line((x0, y0, x1, y1), fill=255,
                      width=rng.randint(1, max(2, min(w, h) // 150)))
        mask_arr = np.asarray(mask_pil.filter(ImageFilter.GaussianBlur(1.0)),
                              dtype=np.float32) / 255.0

    elif motif in {'stain', 'fuzzy', 'localized'}:
        noise    = _fractal_noise_2d(h, w, octaves=4,
                                     base_scale=rng.uniform(0.05, 0.20), rng=rng)
        thr      = rng.uniform(0.45, 0.70)
        sigma    = 3.0 if motif == 'fuzzy' else 1.5
        mask_pil = Image.fromarray(((noise > thr).astype(np.uint8) * 255))
        mask_arr = np.asarray(mask_pil.filter(ImageFilter.GaussianBlur(sigma)),
                              dtype=np.float32) / 255.0

    elif motif in {'indent', 'protrusion'}:
        mask_pil = Image.new('L', (w, h), 0)
        _draw_random_blob(ImageDraw.Draw(mask_pil), w, h, rng)
        mask_arr = np.asarray(mask_pil.filter(ImageFilter.GaussianBlur(4.0)),
                              dtype=np.float32) / 255.0

    elif motif == 'crack':
        mask_pil = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask_pil)
        pts = [(rng.randint(0, w-1), rng.randint(0, h-1))]
        for _ in range(rng.randint(3, 8)):
            x = int(np.clip(pts[-1][0] + rng.randint(-w//6, w//6), 0, w-1))
            y = int(np.clip(pts[-1][1] + rng.randint(-h//6, h//6), 0, h-1))
            pts.append((x, y))
        draw.line(pts, fill=255, width=rng.randint(1, 2))
        mask_arr = np.asarray(mask_pil.filter(ImageFilter.GaussianBlur(1.0)),
                              dtype=np.float32) / 255.0

    elif motif == 'fragment':
        mask_pil = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask_pil)
        x0, y0 = rng.randint(0, w-1), rng.randint(0, h-1)
        bw, bh = rng.randint(w//20, w//6), rng.randint(h//20, h//6)
        draw.rectangle((x0, y0, min(w-1, x0+bw), min(h-1, y0+bh)), fill=255)
        mask_arr = np.asarray(mask_pil, dtype=np.float32) / 255.0

    elif motif == 'hole':
        mask_pil = Image.new('L', (w, h), 0)
        draw = ImageDraw.Draw(mask_pil)
        for _ in range(rng.randint(3, 20)):
            r  = rng.randint(1, max(2, min(w, h) // 120))
            cx, cy = rng.randint(0, w-1), rng.randint(0, h-1)
            draw.ellipse((cx-r, cy-r, cx+r, cy+r), fill=255)
        mask_arr = np.asarray(mask_pil, dtype=np.float32) / 255.0

    else:
        noise = _fractal_noise_2d(h, w, rng=rng)
        mask_arr = (noise > 0.6).astype(np.float32)

    mask_arr = np.clip(mask_arr, 0.0, 1.0)
    img_np   = np.asarray(img.convert('RGB'), dtype=np.float32)

    if motif in {'scratch', 'crack', 'hole', 'fragment'}:
        effect = img_np * (1.0 - rng.uniform(0.25, 0.65))
    elif motif in {'stain', 'localized'}:
        color  = np.array([rng.uniform(40, 180)] * 3, dtype=np.float32)
        alpha  = rng.uniform(0.25, 0.6)
        effect = (1 - alpha) * img_np + alpha * color
    elif motif == 'fuzzy':
        color  = np.array([rng.uniform(60,160), rng.uniform(120,220),
                           rng.uniform(60,160)], dtype=np.float32)
        alpha  = rng.uniform(0.25, 0.55)
        effect = (1 - alpha) * img_np + alpha * color
    else:
        effect = np.clip(img_np + 255.0 * rng.uniform(-0.25, 0.25), 0, 255)

    out = img_np * (1 - mask_arr[..., None]) + effect * mask_arr[..., None]
    return (Image.fromarray(np.clip(out, 0, 255).astype(np.uint8)),
            Image.fromarray((mask_arr * 255).astype(np.uint8)))


## Dataset

In [ ]:
class ADLSegmentationDataset(Dataset):
    def __init__(self, samples: list, image_size: int = 256, train: bool = True,
                 p_synth: float = 0.30, p_cutpaste: float = 0.20,
                 motifs: list[str] | None = None,
                 crop_bank: AnomalyCropBank | None = None, seed: int = 0,
                 pose_lookup: dict | None = None):
        self.samples    = samples
        self.image_size = image_size
        self.train      = train
        self.p_synth    = p_synth
        self.p_cutpaste = p_cutpaste
        self.motifs     = motifs or ['scratch', 'stain', 'localized']
        self.crop_bank  = crop_bank
        self.seed       = seed
        self.pose_lookup = pose_lookup or {}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        s   = self.samples[idx]
        # Training: unseeded so each epoch sees different augmentations.
        # Validation: fixed seed per sample for reproducible eval.
        rng = random.Random() if self.train else random.Random(self.seed + idx)

        img  = Image.open(s.image_path).convert('RGB').resize(
                   (self.image_size, self.image_size), Image.BILINEAR)
        mask = (Image.open(s.mask_path).convert('L').resize(
                   (self.image_size, self.image_size), Image.NEAREST)
                if s.mask_path else Image.new('L', (self.image_size, self.image_size), 0))

        img_np  = np.asarray(img,  dtype=np.float32)
        mask_np = (np.asarray(mask, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)

        if self.train and not s.is_anomaly:
            roll = rng.random()
            if (self.crop_bank and len(self.crop_bank) > 0
                    and roll < self.p_cutpaste):
                src_img, src_mask = self.crop_bank.sample(rng)
                img_np, mask_np = apply_cutpaste(img_np, mask_np, src_img, src_mask, rng)
            elif roll < self.p_cutpaste + self.p_synth:
                motif   = rng.choice(self.motifs)
                pil_tmp = Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))
                pil_out, syn_pil = synthesize_anomaly(pil_tmp, motif, rng)
                img_np  = np.asarray(pil_out, dtype=np.float32)
                syn_np  = (np.asarray(syn_pil, dtype=np.float32) / 255.0 > 0.5).astype(np.float32)
                mask_np = np.maximum(mask_np, syn_np)

        if self.train:
            # Gaussian blur
            if rng.random() < 0.5:
                pil_blur = Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))
                pil_blur = pil_blur.filter(ImageFilter.GaussianBlur(rng.uniform(0.0, 0.8)))
                img_np   = np.asarray(pil_blur, dtype=np.float32)

            # Horizontal flip — consistent across img and mask
            if rng.random() < P_HFLIP:
                img_np  = img_np[:, ::-1, :].copy()
                mask_np = mask_np[:, ::-1].copy()

            # Vertical flip — consistent across img and mask
            if rng.random() < P_VFLIP:
                img_np  = img_np[::-1, :, :].copy()
                mask_np = mask_np[::-1, :].copy()

            # Color jitter — img only, mask unchanged
            if rng.random() < P_COLOR_JITTER:
                bf = rng.uniform(0.75, 1.25)
                img_np = np.clip(img_np * bf, 0, 255)
                cf = rng.uniform(0.75, 1.25)
                mean_val = img_np.mean()
                img_np = np.clip((img_np - mean_val) * cf + mean_val, 0, 255)
                sf = rng.uniform(0.75, 1.25)
                gray = (0.299*img_np[:,:,0] + 0.587*img_np[:,:,1]
                        + 0.114*img_np[:,:,2])[:, :, None]
                img_np = np.clip(gray + sf * (img_np - gray), 0, 255)

        img_final  = Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))
        mask_final = Image.fromarray((mask_np * 255).astype(np.uint8))

        cluster = self.pose_lookup.get(Path(s.image_path).name, 0)

        return {
            'image':      pil_to_tensor(img_final),
            'mask':       mask_to_tensor(mask_final),
            'foreground': foreground_from_image(img_final),
            'view_id':    torch.tensor(cluster, dtype=torch.long),
            'is_anomaly': torch.tensor(1 if s.is_anomaly else 0, dtype=torch.long),
            'sample_id':  s.sample_id,
            'image_path': s.image_path,
        }


## Model — Attention U-Net + FiLM view conditioning

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    encoder:            str  = 'resnet34'
    pretrained:         bool = True
    view_count:         int  = 8   # max pose clusters across all classes
    view_embed_dim:     int  = 64
    base_channels:      int  = 256
    use_attention_gates: bool = True


class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None):
        super().__init__()
        p = k // 2 if p is None else p
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


class FiLM(nn.Module):
    """Feature-wise Linear Modulation: scales+shifts feature maps per channel."""
    def __init__(self, cond_dim, feat_ch):
        super().__init__()
        self.to_gamma = nn.Linear(cond_dim, feat_ch)
        self.to_beta  = nn.Linear(cond_dim, feat_ch)
    def forward(self, x, cond):
        g = self.to_gamma(cond).unsqueeze(-1).unsqueeze(-1)
        b = self.to_beta(cond).unsqueeze(-1).unsqueeze(-1)
        return x * (1.0 + g) + b


class AttentionGate(nn.Module):
    """
    Soft spatial attention on encoder skip connections (Attention U-Net).
    g  = gating signal  (lower-res decoder feature)
    x  = skip connection (higher-res encoder feature)
    The gate learns which spatial regions of x are relevant given context g.
    """
    def __init__(self, g_ch: int, x_ch: int, inter_ch: int):
        super().__init__()
        self.W_g  = nn.Conv2d(g_ch,    inter_ch, kernel_size=1, bias=True)
        self.W_x  = nn.Conv2d(x_ch,    inter_ch, kernel_size=1, bias=False)
        self.psi  = nn.Conv2d(inter_ch, 1,        kernel_size=1, bias=True)

    def forward(self, g: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        g_up = F.interpolate(self.W_g(g), size=x.shape[-2:],
                             mode='bilinear', align_corners=False)
        att  = torch.sigmoid(self.psi(F.relu(g_up + self.W_x(x), inplace=True)))
        return x * att


class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, cond_dim,
                 use_attention: bool = True):
        super().__init__()
        self.attn = (
            AttentionGate(g_ch=in_ch, x_ch=skip_ch,
                          inter_ch=max(1, skip_ch // 2))
            if use_attention else None
        )
        self.conv1 = ConvBNAct(in_ch + skip_ch, out_ch)
        self.conv2 = ConvBNAct(out_ch, out_ch)
        self.film  = FiLM(cond_dim, out_ch)

    def forward(self, x, skip, cond):
        if self.attn is not None:
            skip = self.attn(x, skip)
        x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
        x = torch.cat([x, skip], dim=1)
        x = self.conv1(x)
        x = self.film(x, cond)
        x = self.conv2(x)
        return x


class ResNetEncoder(nn.Module):
    def __init__(self, name: str = 'resnet34', pretrained: bool = True):
        super().__init__()
        if name == 'resnet18':
            m = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
            self.out_channels = (64, 64, 128, 256, 512)
        elif name == 'resnet34':
            m = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
            self.out_channels = (64, 64, 128, 256, 512)
        elif name == 'resnet50':
            m = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
            self.out_channels = (64, 256, 512, 1024, 2048)
        else:
            raise ValueError(f'Unknown encoder: {name}')
        self.stem    = nn.Sequential(m.conv1, m.bn1, m.relu)
        self.maxpool = m.maxpool
        self.layer1  = m.layer1
        self.layer2  = m.layer2
        self.layer3  = m.layer3
        self.layer4  = m.layer4

    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.layer1(self.maxpool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        return x0, x1, x2, x3, x4


class MultiViewUNet(nn.Module):
    def __init__(self, cfg: ModelConfig = ModelConfig()):
        super().__init__()
        self.cfg     = cfg
        self.encoder = ResNetEncoder(cfg.encoder, cfg.pretrained)
        enc0, enc1, enc2, enc3, enc4 = self.encoder.out_channels
        dec = cfg.base_channels

        self.view_embed = nn.Embedding(cfg.view_count, cfg.view_embed_dim)
        self.cond_mlp   = nn.Sequential(
            nn.Linear(cfg.view_embed_dim, cfg.view_embed_dim), nn.SiLU(inplace=True),
            nn.Linear(cfg.view_embed_dim, cfg.view_embed_dim),
        )
        cond_dim = cfg.view_embed_dim
        ua = cfg.use_attention_gates

        self.bottleneck = nn.Sequential(ConvBNAct(enc4, dec), ConvBNAct(dec, dec))
        self.up3 = UpBlock(dec,     enc3, dec//2, cond_dim, ua)
        self.up2 = UpBlock(dec//2,  enc2, dec//4, cond_dim, ua)
        self.up1 = UpBlock(dec//4,  enc1, dec//8, cond_dim, ua)
        self.up0 = UpBlock(dec//8,  enc0, dec//8, cond_dim, ua)
        self.head = nn.Sequential(
            ConvBNAct(dec//8, dec//16),
            nn.Conv2d(dec//16, 1, kernel_size=1),
        )

    def forward(self, x: torch.Tensor, view_id: torch.Tensor) -> torch.Tensor:
        cond = self.cond_mlp(self.view_embed(view_id.long()))
        x0, x1, x2, x3, x4 = self.encoder(x)
        b  = self.bottleneck(x4)
        d3 = self.up3(b,  x3, cond)
        d2 = self.up2(d3, x2, cond)
        d1 = self.up1(d2, x1, cond)
        d0 = self.up0(d1, x0, cond)
        d0 = F.interpolate(d0, scale_factor=2.0, mode='bilinear', align_corners=False)
        return self.head(d0)


## Loss & metrics

In [ ]:
def focal_bce_loss(logits, targets, gamma=2.0):
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    pt  = torch.exp(-bce)
    return ((1 - pt) ** gamma * bce).mean()

def dice_loss(logits, targets, eps=1e-6):
    p = torch.sigmoid(logits).view(logits.shape[0], -1)
    t = targets.view(targets.shape[0], -1)
    inter = (p * t).sum(1)
    return 1.0 - ((2 * inter + eps) / (p.sum(1) + t.sum(1) + eps)).mean()

def bce_dice_loss(logits, targets, foreground_mask=None, bce_weight=0.5):
    if foreground_mask is not None:
        logits  = logits  * foreground_mask
        targets = targets * foreground_mask
    bce = (focal_bce_loss(logits, targets, FOCAL_GAMMA)
           if USE_FOCAL_LOSS else
           F.binary_cross_entropy_with_logits(logits, targets))
    return bce_weight * bce + (1 - bce_weight) * dice_loss(logits, targets)

@torch.no_grad()
def dice_score_from_logits(logits, targets, eps=1e-6):
    preds = (torch.sigmoid(logits) > 0.5).float().view(logits.shape[0], -1)
    t     = targets.view(targets.shape[0], -1)
    return ((2 * (preds * t).sum(1) + eps) / (preds.sum(1) + t.sum(1) + eps)).mean()


## Training

In [ ]:
def _worker_init_fn(worker_id):
    random.seed(torch.initial_seed() % (2**31))


def _build_pose_lookup(pose_assignments: dict | None) -> dict:
    lookup = {}
    if not pose_assignments:
        return lookup
    for path, label in zip(pose_assignments['train_paths'], pose_assignments['train_labels']):
        lookup[os.path.basename(path)] = int(label)
    for path, label in zip(pose_assignments['test_paths'], pose_assignments['test_labels']):
        lookup[os.path.basename(path)] = int(label)
    return lookup


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    losses, dices = [], []
    for batch in loader:
        x, y, fg, vid = (batch[k].to(device) for k in
                         ('image', 'mask', 'foreground', 'view_id'))
        logits = model(x, vid)
        losses.append(bce_dice_loss(logits, y, fg).item())
        dices.append(dice_score_from_logits(logits * fg, y * fg).item())
    return {'loss': float(np.mean(losses)), 'dice': float(np.mean(dices))}


def train_one_class(
    dataset_dir, class_name, desc_csv, out_dir,
    image_size=224, batch_size=32, epochs=25, lr=3e-4,
    val_ratio=0.15, p_synth=0.4, p_cutpaste=0.3, seed=42,
    encoder='resnet34', use_attention_gates=True,
    pose_assignments=None, lr_warmup_epochs=2,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    desc   = load_anomaly_descriptions(desc_csv)
    motifs = build_motifs_for_class(desc.get(class_name))
    print(f'  motifs: {motifs}')

    pose_lookup = _build_pose_lookup(pose_assignments)
    print(f'  pose lookup: {len(pose_lookup)} entries'
          + (f'  (k={pose_assignments["k"]})' if pose_assignments else ' (no data → cluster 0)'))

    samples        = scan_class_dataset(dataset_dir, class_name)
    train_s, val_s = split_by_sample_id(samples, val_ratio, seed)
    crop_bank      = AnomalyCropBank(train_s, image_size)

    train_ds = ADLSegmentationDataset(
        train_s, image_size, train=True,
        p_synth=p_synth, p_cutpaste=p_cutpaste,
        motifs=motifs, crop_bank=crop_bank, seed=seed,
        pose_lookup=pose_lookup)
    val_ds = ADLSegmentationDataset(
        val_s, image_size, train=False, seed=seed,
        pose_lookup=pose_lookup)

    train_loader = DataLoader(train_ds, batch_size, shuffle=True,
                              num_workers=2, pin_memory=True,
                              worker_init_fn=_worker_init_fn,
                              persistent_workers=True)
    val_loader   = DataLoader(val_ds, batch_size, shuffle=False,
                              num_workers=2, pin_memory=True,
                              persistent_workers=True)

    cfg   = ModelConfig(encoder=encoder, pretrained=True,
                        view_count=POSE_VIEW_COUNT,
                        use_attention_gates=use_attention_gates)
    model = MultiViewUNet(cfg).to(device)
    opt    = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    total_steps  = epochs * len(train_loader)
    warmup_steps = lr_warmup_epochs * len(train_loader)
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    outp = Path(out_dir) / class_name
    outp.mkdir(parents=True, exist_ok=True)

    best_path = str(outp / 'best.pt')
    best_val  = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        pbar = tqdm(train_loader, desc=f'{class_name} {epoch}/{epochs}')
        for batch in pbar:
            x, y, fg, vid = (batch[k].to(device, non_blocking=True)
                             for k in ('image', 'mask', 'foreground', 'view_id'))
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                loss = bce_dice_loss(model(x, vid), y, fg)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            scheduler.step()
            pbar.set_postfix(loss=f'{loss.item():.4f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}')

        m = evaluate(model, val_loader, device)
        with (outp / 'metrics.csv').open('a') as f:
            f.write(f"{epoch},{m['loss']:.6f},{m['dice']:.6f}\n")
        if m['loss'] < best_val:
            best_val = m['loss']
            torch.save({'model': model.state_dict(), 'cfg': asdict(cfg)}, best_path)

    return best_path


def train_all_classes(dataset_dir, desc_csv, out_dir, pose_data=None, **kw):
    pose_data = pose_data or {}
    for name in sorted(os.listdir(dataset_dir)):
        if name.startswith('class_'):
            print(f'\n=== Training {name} ===')
            train_one_class(dataset_dir=dataset_dir, class_name=name,
                            desc_csv=desc_csv, out_dir=out_dir,
                            pose_assignments=pose_data.get(name),
                            **kw)


### Run training

In [ ]:
_train_kw = dict(
    dataset_dir=DATASET_DIR, desc_csv=DESC_CSV, out_dir=SUBMISSION_MODEL_DIR,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
    val_ratio=VAL_RATIO, p_synth=P_SYNTH, p_cutpaste=P_CUTPASTE, seed=SEED,
    encoder=ENCODER, use_attention_gates=USE_ATTENTION_GATES,
    lr_warmup_epochs=LR_WARMUP_EPOCHS,
)

if CLASS_NAME is None:
    train_all_classes(pose_data=POSE_DATA, **_train_kw)
else:
    train_one_class(class_name=CLASS_NAME,
                    pose_assignments=POSE_DATA.get(CLASS_NAME),
                    **_train_kw)



=== Training class_01 ===
  motifs: ['fragment', 'indent', 'localized', 'scratch', 'stain']
  pose lookup: 3065 entries  (k=4)
  AnomalyCropBank: 15 crops
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 238MB/s]
class_01 25/25: 100%|██████████| 70/70 [00:43<00:00,  1.59it/s, loss=0.3343, lr=0.00e+00]



=== Training class_02 ===
  motifs: ['fragment', 'localized', 'protrusion', 'scratch', 'stain']
  pose lookup: 2935 entries  (k=2)
  AnomalyCropBank: 21 crops


class_02 25/25: 100%|██████████| 58/58 [00:38<00:00,  1.49it/s, loss=0.3449, lr=0.00e+00]



=== Training class_03 ===
  motifs: ['fragment', 'localized', 'scratch', 'stain']
  pose lookup: 3310 entries  (k=3)
  AnomalyCropBank: 10 crops


...

class_06 25/25: 100%|██████████| 63/63 [00:55<00:00,  1.14it/s, loss=0.2944, lr=0.00e+00]



=== Training class_07 ===
  motifs: ['crack', 'fragment', 'hole', 'localized']
  pose lookup: 3060 entries  (k=2)
  AnomalyCropBank: 23 crops


class_07 25/25: 100%|██████████| 58/58 [00:32<00:00,  1.78it/s, loss=0.3623, lr=0.00e+00]



=== Training class_08 ===
  motifs: ['indent', 'scratch', 'crack', 'fragment', 'localized', 'stain']
  pose lookup: 3005 entries  (k=8)
  AnomalyCropBank: 18 crops


class_08 25/25: 100%|██████████| 56/56 [00:34<00:00,  1.64it/s, loss=0.4267, lr=0.00e+00]


## Inference — TTA + multi-view aggregation

In [ ]:
@torch.no_grad()
def load_model(ckpt_path: str, device: torch.device) -> MultiViewUNet:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    cfg  = ModelConfig(**ckpt.get('cfg', {}))
    m    = MultiViewUNet(cfg)
    m.load_state_dict(ckpt['model'], strict=True)
    return m.to(device).eval()


@torch.no_grad()
def _infer_single_tta(model, img_rs: Image.Image, view_id_t: torch.Tensor,
                      device, use_tta: bool = True) -> np.ndarray:
    flips = ['none', 'h', 'v'] if use_tta else ['none']
    preds = []
    for flip in flips:
        if   flip == 'h': img_f = img_rs.transpose(Image.FLIP_LEFT_RIGHT)
        elif flip == 'v': img_f = img_rs.transpose(Image.FLIP_TOP_BOTTOM)
        else:             img_f = img_rs
        x    = pil_to_tensor(img_f).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(x, view_id_t))[0, 0].cpu().numpy()
        if   flip == 'h': prob = prob[:, ::-1].copy()
        elif flip == 'v': prob = prob[::-1, :].copy()
        preds.append(prob)
    return np.mean(preds, axis=0)


def _aggregate_views(probs: list[np.ndarray],
                     boost_threshold: float = 0.25) -> list[np.ndarray]:
    max_conf = max(float(p.max()) for p in probs)
    if max_conf <= boost_threshold:
        return probs
    factor = 1.0 + 0.5 * (max_conf - boost_threshold) / (1.0 - boost_threshold)
    return [np.clip(p * factor, 0.0, 1.0) for p in probs]


def infer_all_classes(
    dataset_dir, ckpt_dir, out_dir,
    image_size=256, class_thresholds=None,
    use_tta=True, multiview_aggregate=True,
    pose_data=None,
):
    device           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    class_thresholds = class_thresholds or {}
    pose_data        = pose_data or {}

    for class_name in sorted(os.listdir(dataset_dir)):
        if not class_name.startswith('class_'):
            continue
        ckpt_path = Path(ckpt_dir) / class_name / 'best.pt'
        if not ckpt_path.exists():
            print(f'[skip] missing {ckpt_path}'); continue

        model       = load_model(str(ckpt_path), device)
        pose_lookup = _build_pose_lookup(pose_data.get(class_name))
        test_dir    = Path(dataset_dir) / class_name / 'test'
        class_out   = Path(out_dir) / class_name
        class_out.mkdir(parents=True, exist_ok=True)

        stem_files: dict[str, list[str]] = defaultdict(list)
        for fn in sorted(os.listdir(test_dir)):
            if fn.endswith('.png'):
                stem_files[_parse_view_and_id(fn)[0]].append(fn)

        for stem, fns in tqdm(stem_files.items(), desc=f'infer {class_name}'):
            view_results: list[tuple[str, np.ndarray, tuple]] = []

            for fn in sorted(fns):
                _, view_id = _parse_view_and_id(fn)
                img        = Image.open(str(test_dir / fn)).convert('RGB')
                orig_size  = img.size
                img_rs     = img.resize((image_size, image_size), Image.BILINEAR)
                fg         = foreground_from_image(img_rs)[0].numpy()
                cluster    = pose_lookup.get(fn, 0)
                v_t        = torch.tensor([cluster], dtype=torch.long, device=device)
                prob       = _infer_single_tta(model, img_rs, v_t, device, use_tta)
                prob       = prob * fg
                view_results.append((fn, prob, orig_size))

            if multiview_aggregate:
                raw          = _aggregate_views([r[1] for r in view_results])
                view_results = [(fn, p, sz)
                                for (fn, _, sz), p in zip(view_results, raw)]

            for fn, prob, orig_size in view_results:
                stem_fn, view_id = _parse_view_and_id(fn)
                out_fn = f'{stem_fn}_view{view_id + 1}.png'
                Image.fromarray((prob * 255).astype(np.uint8)).resize(
                    orig_size, Image.BILINEAR).save(str(class_out / out_fn))


### Run inference

In [ ]:
infer_all_classes(
    dataset_dir=DATASET_DIR, ckpt_dir=SUBMISSION_MODEL_DIR, out_dir=PRED_DIR,
    image_size=IMAGE_SIZE, class_thresholds=CLASS_THRESHOLDS,
    use_tta=USE_TTA, multiview_aggregate=MULTIVIEW_AGGREGATE,
    pose_data=POSE_DATA,
)
print('Done. Predictions in:', PRED_DIR)


infer class_08: 100%|██████████| 192/192 [01:20<00:00,  2.38it/s]

Done. Predictions in: /content/drive/MyDrive/ADL/challenge/preds_v3_fast


## Generate submission CSV

In [ ]:
def float_matrix_to_q8rle(x: np.ndarray) -> str:
    q    = np.clip(np.rint(np.asarray(x, dtype=np.float32) * 255), 0, 255).astype(np.uint8)
    h, w = q.shape
    flat = q.T.reshape(-1)
    if flat.size == 0:
        return f"q8rle {h} {w}"
    cuts   = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    starts = np.r_[0, cuts]; ends = np.r_[cuts, flat.size]
    parts  = ["q8rle", str(h), str(w)]
    for v, n in zip(flat[starts], ends - starts):
        parts += [str(int(v)), str(int(n))]
    return " ".join(parts)


submission_dir  = Path('/content/drive/MyDrive/ADL/challenge/submissions')
submission_dir.mkdir(parents=True, exist_ok=True)
submission_path = submission_dir / 'submission_v4.csv'
results = []
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for class_name in sorted(os.listdir(DATASET_DIR)):
    if not class_name.startswith('class_'):
        continue
    ckpt_path = Path(SUBMISSION_MODEL_DIR) / class_name / 'best.pt'
    if not ckpt_path.exists():
        print(f'[skip] {class_name}'); continue

    model       = load_model(str(ckpt_path), device)
    pose_lookup = _build_pose_lookup(POSE_DATA.get(class_name))
    test_dir    = Path(DATASET_DIR) / class_name / 'test'

    stem_files: dict[str, list[str]] = defaultdict(list)
    for fn in sorted(os.listdir(test_dir)):
        if fn.endswith('.png'):
            stem_files[_parse_view_and_id(fn)[0]].append(fn)

    for stem, fns in tqdm(stem_files.items(), desc=f'submit {class_name}'):
        view_results = []
        for fn in sorted(fns):
            _, view_id = _parse_view_and_id(fn)
            img        = Image.open(str(test_dir / fn)).convert('RGB')
            orig_size  = img.size
            img_rs     = img.resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
            fg         = foreground_from_image(img_rs)[0].numpy()
            cluster    = pose_lookup.get(fn, 0)
            v_t        = torch.tensor([cluster], dtype=torch.long, device=device)
            prob       = _infer_single_tta(model, img_rs, v_t, device, USE_TTA)
            prob       = prob * fg
            view_results.append((fn, prob, orig_size))

        if MULTIVIEW_AGGREGATE:
            raw          = _aggregate_views([r[1] for r in view_results])
            view_results = [(fn, p, sz) for (fn, _, sz), p in zip(view_results, raw)]

        for fn, prob, orig_size in view_results:
            stem_fn, view_id = _parse_view_and_id(fn)
            prob_pil  = Image.fromarray((prob * 255).astype(np.uint8)).resize(
                            orig_size, Image.BILINEAR)
            prob_np   = np.asarray(prob_pil, dtype=np.float32) / 255.0
            image_id  = Path(fn).stem
            results.append({'ID': image_id, 'Label': float_matrix_to_q8rle(prob_np)})

print(f'Writing {len(results)} entries to {submission_path}')
with open(submission_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ID', 'Label'])
    w.writeheader(); w.writerows(results)
print('Done.')


submit class_08: 100%|██████████| 192/192 [00:54<00:00,  3.52it/s]


Writing 5910 entries to /content/drive/MyDrive/ADL/challenge/submissions/submission_v3.csv
Done.
